# Flight Delay Prediction with KNN

## Data Loading

Load the raw flights dataset and take an initial look at its shape, columns, and missing values.

In [ ]:
import pandas as pd

df = pd.read_csv('../data/flights.csv')
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.columns.tolist()

In [ ]:
# Check missing values in key columns
print(df[['ARR_DELAY', 'CANCELLED', 'DIVERTED']].isnull().sum())

# Check how many flights were cancelled/diverted
print(df['CANCELLED'].value_counts())
print(df['DIVERTED'].value_counts())

## Data Cleaning

Keep only completed flights (not cancelled or diverted), then build the binary `DELAYED` target (arrival 15+ minutes late).

In [ ]:
# Step 1: Keep only completed flights (not cancelled, not diverted)
df_clean = df[(df['CANCELLED'] == 0) & (df['DIVERTED'] == 0)].copy()

print("Rows before cleaning:", df.shape[0])
print("Rows after cleaning:", df_clean.shape[0])

In [51]:
# Step 2: Create binary target column
# Standard definition: a flight is "delayed" if it arrives 15+ minutes late
df_clean['DELAYED'] = (df_clean['ARR_DELAY'] >= 15).astype(int)

# Check the balance of the target
print(df_clean['DELAYED'].value_counts())
print(df_clean['DELAYED'].value_counts(normalize=True))

DELAYED
0    2379941
1     533863
Name: count, dtype: int64
DELAYED
0    0.816781
1    0.183219
Name: proportion, dtype: float64


## Feature Engineering

Select pre-departure features (to avoid data leakage) and derive day-of-week from the flight date.

In [52]:
# Step 3: Select only pre-departure features (no data leakage)
features = ['AIRLINE', 'ORIGIN', 'DEST', 'CRS_DEP_TIME', 'CRS_ARR_TIME', 
            'CRS_ELAPSED_TIME', 'DISTANCE']

target = 'DELAYED'

# Also grab FL_DATE so we can extract day-of-week
df_model = df_clean[features + ['FL_DATE', target]].copy()

df_model.head()

,AIRLINE,ORIGIN,DEST,CRS_DEP_TIME,CRS_ARR_TIME,CRS_ELAPSED_TIME,DISTANCE,FL_DATE,DELAYED
0,United Air Lines Inc.,FLL,EWR,1155,1501,186.0,1065.0,2019-01-09,0
1,Delta Air Lines Inc.,MSP,SEA,2120,2315,235.0,1399.0,2022-11-19,0
2,United Air Lines Inc.,DEN,MSP,954,1252,118.0,680.0,2022-07-22,0
3,Delta Air Lines Inc.,MSP,SFO,1609,1829,260.0,1589.0,2023-03-06,1
4,Spirit Air Lines,MCO,DFW,1840,2041,181.0,985.0,2020-02-23,0


In [ ]:
# Step 4: Extract day of week from FL_DATE
df_model['FL_DATE'] = pd.to_datetime(df_model['FL_DATE'])
df_model['DAY_OF_WEEK'] = df_model['FL_DATE'].dt.dayofweek

# Drop the raw date now that we've extracted what we need
df_model = df_model.drop(columns=['FL_DATE'])

df_model.head()

In [ ]:
print("Airlines:", df_model['AIRLINE'].nunique())
print("Origins:", df_model['ORIGIN'].nunique())
print("Destinations:", df_model['DEST'].nunique())

## Encoding

One-hot encode `AIRLINE` and frequency-encode the high-cardinality `ORIGIN` and `DEST` columns.

In [56]:
# Step 5: Encode categorical columns

# One-hot encode AIRLINE (18 categories - manageable)
df_model = pd.get_dummies(df_model, columns=['AIRLINE'], drop_first=True)

# Frequency encode ORIGIN and DEST (380 categories each - too many for one-hot)
origin_freq = df_model['ORIGIN'].value_counts()
dest_freq = df_model['DEST'].value_counts()

df_model['ORIGIN_FREQ'] = df_model['ORIGIN'].map(origin_freq)
df_model['DEST_FREQ'] = df_model['DEST'].map(dest_freq)

# Drop the original text columns now that we've encoded them
df_model = df_model.drop(columns=['ORIGIN', 'DEST'])

print(df_model.shape)
df_model.head()

(2913804, 25)


,CRS_DEP_TIME,CRS_ARR_TIME,CRS_ELAPSED_TIME,DISTANCE,DELAYED,DAY_OF_WEEK,AIRLINE_Allegiant Air,AIRLINE_American Airlines Inc.,AIRLINE_Delta Air Lines Inc.,AIRLINE_Endeavor Air Inc.,...,AIRLINE_JetBlue Airways,AIRLINE_Mesa Airlines Inc.,AIRLINE_PSA Airlines Inc.,AIRLINE_Republic Airline,AIRLINE_SkyWest Airlines Inc.,AIRLINE_Southwest Airlines Co.,AIRLINE_Spirit Air Lines,AIRLINE_United Air Lines Inc.,ORIGIN_FREQ,DEST_FREQ
0,1155,1501,186.0,1065.0,0,2,False,False,False,False,...,False,False,False,False,False,False,False,True,39093,50560
1,2120,2315,235.0,1399.0,0,5,False,False,True,False,...,False,False,False,False,False,False,False,False,58886,69502
2,954,1252,118.0,680.0,0,4,False,False,False,False,...,False,False,False,False,False,False,False,True,116362,58570
3,1609,1829,260.0,1589.0,1,0,False,False,True,False,...,False,False,False,False,False,False,False,False,58886,57575
4,1840,2041,181.0,985.0,0,6,False,False,False,False,...,False,False,False,False,False,False,True,False,62027,125047


## Sampling, Train/Test Split & Scaling

Separate features and target, take a stratified 40k-row sample, split 80/20, and scale features with `MinMaxScaler`.

In [57]:
from sklearn.preprocessing import MinMaxScaler

# Separate features (X) from target (y)
X = df_model.drop(columns=['DELAYED'])
y = df_model['DELAYED']

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (2913804, 24)
y shape: (2913804,)


In [58]:
from sklearn.model_selection import train_test_split

# Sample down to 40,000 rows, preserving the class balance
X_sample, _, y_sample, _ = train_test_split(
    X, y,
    train_size=40000,
    stratify=y,
    random_state=42
)

print("Sampled X shape:", X_sample.shape)
print(y_sample.value_counts(normalize=True))

Sampled X shape: (40000, 24)
DELAYED
0    0.816775
1    0.183225
Name: proportion, dtype: float64


In [59]:
# Step: Train/test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X_sample, y_sample,
    test_size=0.2,
    stratify=y_sample,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (32000, 24)
X_test shape: (8000, 24)


In [60]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling complete.")

Scaling complete.


## Baseline KNN

Train a baseline KNN classifier (k=5) on the imbalanced data and evaluate accuracy, precision, recall, and F1.

In [61]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Train a baseline KNN model with k=5
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

# Make predictions on the test set
y_pred = knn.predict(X_test_scaled)

# Evaluate
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.787
Precision: 0.25413223140495866
Recall: 0.08390177353342429
F1 Score: 0.12615384615384614

Confusion Matrix:
[[6173  361]
 [1343  123]]


## SMOTE

Oversample the minority (delayed) class in the training set with SMOTE, then retrain KNN on the balanced data.

In [62]:
from imblearn.over_sampling import SMOTE

# Apply SMOTE only to training data
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE:", y_train_smote.value_counts().to_dict())

Before SMOTE: {0: 26137, 1: 5863}
After SMOTE: {0: 26137, 1: 26137}


In [63]:
# Train KNN on SMOTE-balanced data
knn_smote = KNeighborsClassifier(n_neighbors=5)
knn_smote.fit(X_train_smote, y_train_smote)

# Predict on the ORIGINAL (untouched) test set
y_pred_smote = knn_smote.predict(X_test_scaled)

# Evaluate
print("=== KNN with SMOTE ===")
print("Accuracy:", accuracy_score(y_test, y_pred_smote))
print("Precision:", precision_score(y_test, y_pred_smote))
print("Recall:", recall_score(y_test, y_pred_smote))
print("F1 Score:", f1_score(y_test, y_pred_smote))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_smote))

=== KNN with SMOTE ===
Accuracy: 0.585875
Precision: 0.20784561847516608
Recall: 0.4481582537517053
F1 Score: 0.2839853036524746

Confusion Matrix:
[[4030 2504]
 [ 809  657]]


## K-Tuning

Use cross-validation and test-set comparisons across a range of `k` values to choose the number of neighbors.

In [ ]:
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt

k_values = range(1, 26, 2)  # test odd k values from 1 to 25
f1_scores = []

for k in k_values:
    knn_temp = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn_temp, X_train_smote, y_train_smote, cv=5, scoring='f1')
    f1_scores.append(scores.mean())
    print(f"k={k}, F1 (CV avg): {scores.mean():.4f}")

# Plot k vs F1 score
plt.figure(figsize=(8,5))
plt.plot(k_values, f1_scores, marker='o')
plt.xlabel('k (number of neighbors)')
plt.ylabel('F1 Score (cross-validated)')
plt.title('KNN: Choosing k using Cross-Validation')
plt.grid(True)
plt.savefig('../outputs/figures/k_tuning.png')
plt.show()

In [ ]:
# Compare KNN on the real test set across a range of k values
for k in [1, 3, 5, 7, 9, 11, 15, 21]:
    knn_k = KNeighborsClassifier(n_neighbors=k)
    knn_k.fit(X_train_smote, y_train_smote)
    pred_k = knn_k.predict(X_test_scaled)  # real test set, not SMOTE
    print(f"k={k} | Accuracy: {accuracy_score(y_test, pred_k):.3f} | "
          f"Precision: {precision_score(y_test, pred_k):.3f} | "
          f"Recall: {recall_score(y_test, pred_k):.3f} | "
          f"F1: {f1_score(y_test, pred_k):.3f}")

## Final KNN

Train the final tuned KNN model (k=9, with SMOTE) and report its metrics.

In [ ]:
# Final tuned KNN model
knn_final = KNeighborsClassifier(n_neighbors=9)
knn_final.fit(X_train_smote, y_train_smote)
y_pred_final = knn_final.predict(X_test_scaled)

print("=== Final Tuned KNN (k=9, with SMOTE) ===")
print("Accuracy:", accuracy_score(y_test, y_pred_final))
print("Precision:", precision_score(y_test, y_pred_final))
print("Recall:", recall_score(y_test, y_pred_final))
print("F1 Score:", f1_score(y_test, y_pred_final))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_final))

## Random Forest Comparison

Compare the tuned KNN against Random Forest baselines using class weighting and SMOTE-balanced data.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)  # Note: original X_train, not scaled/SMOTE - trees don't need scaling

y_pred_rf = rf.predict(X_test)

print("=== Random Forest (class_weight='balanced') ===")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))
print("F1 Score:", f1_score(y_test, y_pred_rf))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

In [ ]:
rf_smote = RandomForestClassifier(n_estimators=100, random_state=42)
rf_smote.fit(X_train_smote, y_train_smote)

y_pred_rf_smote = rf_smote.predict(X_test_scaled)

print("=== Random Forest (with SMOTE) ===")
print("Accuracy:", accuracy_score(y_test, y_pred_rf_smote))
print("Precision:", precision_score(y_test, y_pred_rf_smote))
print("Recall:", recall_score(y_test, y_pred_rf_smote))
print("F1 Score:", f1_score(y_test, y_pred_rf_smote))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf_smote))

## Model Comparison Summary

Collect the metrics from every model trained above into a single side-by-side table. This reuses the predictions already computed in each section (`y_pred`, `y_pred_smote`, `y_pred_final`, `y_pred_rf`, `y_pred_rf_smote`) — no models are retrained here.

In [ ]:
# Side-by-side comparison of all models (reuses existing predictions)
model_predictions = {
    'Baseline KNN (k=5)':       y_pred,
    'KNN + SMOTE (k=5)':        y_pred_smote,
    'Final KNN (k=9, SMOTE)':   y_pred_final,
    'Random Forest (balanced)': y_pred_rf,
    'Random Forest + SMOTE':    y_pred_rf_smote,
}

results_summary = pd.DataFrame(
    [
        {
            'Model': name,
            'Accuracy':  accuracy_score(y_test, preds),
            'Precision': precision_score(y_test, preds),
            'Recall':    recall_score(y_test, preds),
            'F1':        f1_score(y_test, preds),
        }
        for name, preds in model_predictions.items()
    ]
).round(3)

results_summary